In [1]:
import os, pandas as pd

for path in ["data/value_per_person.csv", "data/vulnerability.csv"]:
    print("\n", path, "| exists:", os.path.exists(path))
    if os.path.exists(path):
        df = pd.read_csv(path, keep_default_na=False)
        print("shape:", df.shape)
        print("columns:", [repr(c) for c in df.columns])


 data/value_per_person.csv | exists: True
shape: (217, 11)
columns: ["'iso3'", "'iso2'", "'name'", "'region'", "'income'", "'year'", "'gdp_pc'", "'estimated'", "'alpha'", "'alpha_estimated'", "'value_pc'"]

 data/vulnerability.csv | exists: True
shape: (212, 8)
columns: ["'iso2'", "'iso3'", "'country_name'", "'theta'", "'beta'", "'zeta'", "'alpha'", "'status'"]


In [3]:
import pandas as pd

vpp = pd.read_csv("data/value_per_person.csv", keep_default_na=False, na_values=[""])
print(vpp.shape)
print(vpp.columns.tolist())
print(vpp.head(3))

(217, 11)
['iso3', 'iso2', 'name', 'region', 'income', 'year', 'gdp_pc', 'estimated', 'alpha', 'alpha_estimated', 'value_pc']
  iso3 iso2         name                                             region  \
0  ABW   AW        Aruba                         Latin America & Caribbean    
1  AFG   AF  Afghanistan  Middle East, North Africa, Afghanistan & Pakistan   
2  AGO   AO       Angola                                Sub-Saharan Africa    

                income    year        gdp_pc  estimated   alpha  \
0          High income  2024.0  38590.565029      False  14.341   
1           Low income  2024.0    416.871146      False  15.065   
2  Lower middle income  2025.0   3129.476623      False  14.790   

   alpha_estimated       value_pc  
0            False  553427.293079  
1            False    6280.163819  
2            False   46284.959259  


In [4]:
import pandas as pd

vul = pd.read_csv("data/vulnerability.csv", keep_default_na=False, na_values=[""])
vpp = pd.read_csv("data/value_per_person.csv", keep_default_na=False, na_values=[""])

# Fallback for countries with no α: the median of Table 3 (my assumption; replace it from Appendix I if it lists α)
ALPHA_DEFAULT = 14.6

alpha_map = dict(zip(vul["iso2"], vul["alpha"]))
vpp["alpha"] = vpp["iso2"].map(alpha_map)
vpp["alpha_estimated"] = vpp["alpha"].isna()
vpp["alpha"] = vpp["alpha"].fillna(ALPHA_DEFAULT)

vpp["value_pc"] = vpp["gdp_pc"] * vpp["alpha"]          # economic exposure per person
vpp.to_csv("data/value_per_person.csv", index=False)
print(vpp[["name", "gdp_pc", "alpha", "value_pc"]].head(10))
print("countries using the default alpha:", vpp["alpha_estimated"].sum())

                   name        gdp_pc   alpha       value_pc
0                 Aruba  38590.565029  14.341  553427.293079
1           Afghanistan    416.871146  15.065    6280.163819
2                Angola   3129.476623  14.790   46284.959259
3               Albania  12998.147870  14.591  189655.975565
4               Andorra  54291.502603  18.268  991797.169554
5  United Arab Emirates  50273.512624  14.702  739121.182603
6             Argentina  14898.088594  18.087  269461.728407
7               Armenia   9474.018493  14.702  139287.019884
8        American Samoa  18017.458938  21.363  384906.975301
9   Antigua and Barbuda  24819.245468  15.312  380032.286606
countries using the default alpha: 5


In [5]:
import numpy as np
from scipy.stats import norm

def loss_ratio(mmi, theta, beta):
    s = np.minimum(np.asarray(mmi, dtype=float), 9.0)      # intensity IX and above is combined at IX
    r = norm.cdf(np.log(np.maximum(s, 1e-9) / theta) / beta)
    return np.where(s < 5.0, 0.0, r)                       # the model covers intensity 5.0 to 9.0

if __name__ == "__main__":
    for name, th, be in [("Japan", 10.29, 0.10), ("Turkey", 9.46, 0.10), ("Nigeria", 8.64, 0.10)]:
        print(name, [round(float(loss_ratio(m, th, be)), 4) for m in (5, 6, 7, 8, 9)])

Japan [0.0, 0.0, 0.0001, 0.0059, 0.0902]
Turkey [0.0, 0.0, 0.0013, 0.0468, 0.3091]
Nigeria [0.0, 0.0001, 0.0176, 0.2208, 0.6584]


In [6]:
from scipy.stats import norm

vul = pd.read_csv("data/vulnerability.csv", keep_default_na=False, na_values=[""])
vpp = pd.read_csv("data/value_per_person.csv", keep_default_na=False, na_values=[""])

# Fallback parameters: medians of Table 4 (my assumption; replace them from Appendix I)
DEFAULT = {"theta": 9.6, "beta": 0.10, "zeta": 1.64}

def place_array(df, col, default):
    s = pop["country"].map(dict(zip(df["iso2"], df[col])))
    return s.fillna(default).values.astype(float)

THETA = place_array(vul, "theta", DEFAULT["theta"])
BETA = place_array(vul, "beta", DEFAULT["beta"])
VALUE_PC = place_array(vpp, "value_pc", float(vpp["value_pc"].median()))
ZETA_BY_COUNTRY = dict(zip(vul["iso2"], vul["zeta"]))

def estimate_loss(lat, lon, ring_mmi, rings=RINGS):
    """ring_mmi: one shaking intensity per ring, in the same order as RINGS."""
    d = distance_km(lat, lon)
    nearest_country = pop["country"].values[np.argmin(d)]
    per_ring, total = [], 0.0
    for (lo, hi), mmi in zip(rings, ring_mmi):
        m = (d >= lo) & (d < hi)
        if not m.any() or mmi < 5.0:
            per_ring.append(0.0); continue
        r = norm.cdf(np.log(min(mmi, 9.0) / THETA[m]) / BETA[m])
        loss = float((P[m] * VALUE_PC[m] * r).sum())
        per_ring.append(loss); total += loss
    zeta = float(ZETA_BY_COUNTRY.get(nearest_country, DEFAULT["zeta"]))
    return {"base": total, "low": total * np.exp(-zeta), "high": total * np.exp(zeta),
            "zeta": zeta, "country": nearest_country, "per_ring": per_ring}

if __name__ == "__main__":
    # The intensities below are MADE-UP inputs, only to confirm the function runs
    print(estimate_loss(35.76, 129.19, [9.0, 8.0, 7.0, 6.0]))

NameError: name 'pop' is not defined